<a href="https://colab.research.google.com/github/GilbertOwen/reefscan-id/blob/e4/notebooks/10_e4_end_to_end.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ReefScan-ID: E4, end to end

The cascade (stage 1 SegFormer-B0 with 3 classes, then stage 2 EfficientNet-B0 on patches) against the direct 5-class SegFormer-B0, scored on live cover (T_live), bleached share (P_bleached) and the KepMenLH category from the same points on every photo. Section 1 runs the whole protocol on the validation photos, so that the code and the protocol are checked first. Run in Google Colab with a GPU runtime.

### Setup: Colab runtime and Google Drive

In [1]:
import os
import torch
from google.colab import drive

drive.mount("/content/drive")

device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("GPU: {}".format(gpu_name))
print("CPU count: {}".format(os.cpu_count()))

base_dir = "/content/drive/MyDrive/ReefScan-ID"
print("Drive folder found: {}".format(os.path.isdir(base_dir)))

Mounted at /content/drive
GPU: Tesla T4
CPU count: 2
Drive folder found: True


### Setup: libraries

In [2]:
!pip install -q datasets
import random
import time
import numpy as np
from PIL import Image

import datasets
print("torch version: {}".format(torch.__version__))
print("datasets version: {}".format(datasets.__version__))
print("numpy version: {}".format(np.__version__))

torch version: 2.14.1+cu130
datasets version: 4.8.5
numpy version: 2.4.6


### Settings

Change these before running if needed. The same points are used for the ground truth and for both methods on every photo.

In [3]:
stage2_source = "both"
points_per_photo = 30
grid_rows = 5
grid_cols = 6
point_seed = 0

patch_crop = 128
patch_size = 224
val_height = 512
val_width = 1024
photo_height = 1024
photo_width = 2048

kepmenlh_bands = [("Buruk", 0.0, 25.0), ("Sedang", 25.0, 50.0), ("Baik", 50.0, 75.0), ("Baik sekali", 75.0, 100.0)]
expected_epochs = {"stage 1 SegFormer-B0, 3 classes": 37, "stage 2 EfficientNet-B0": 9, "direct SegFormer-B0, 5 classes": 25}
output_name = "validation_points.csv"

image_mean_values = [0.485, 0.456, 0.406]
image_std_values = [0.229, 0.224, 0.225]

print("Stage 2 source: {} -> Checkpoints/stage2_efficientnet_b0/{}/best.pt".format(stage2_source, stage2_source))
print("Points per photo: {} = {} rows x {} columns, one random point per cell, seed {}".format(
    points_per_photo, grid_rows, grid_cols, point_seed))
print("Grid size equals the points per photo: {}".format(grid_rows * grid_cols == points_per_photo))
print("Patch: {} px cut from the full photo, resized to {} px".format(patch_crop, patch_size))
print("Segmentation input: photo resized to {} by {}, prediction upsampled to {} by {}".format(
    val_height, val_width, photo_height, photo_width))
print("Points with an ignored true label are dropped for the ground truth and for both methods")
print("KepMenLH bands on percent live cover: {}".format(kepmenlh_bands))
print("Expected epochs in the checkpoints: {}".format(expected_epochs))

Stage 2 source: both -> Checkpoints/stage2_efficientnet_b0/both/best.pt
Points per photo: 30 = 5 rows x 6 columns, one random point per cell, seed 0
Grid size equals the points per photo: True
Patch: 128 px cut from the full photo, resized to 224 px
Segmentation input: photo resized to 512 by 1024, prediction upsampled to 1024 by 2048
Points with an ignored true label are dropped for the ground truth and for both methods
KepMenLH bands on percent live cover: [('Buruk', 0.0, 25.0), ('Sedang', 25.0, 50.0), ('Baik', 50.0, 75.0), ('Baik sekali', 75.0, 100.0)]
Expected epochs in the checkpoints: {'stage 1 SegFormer-B0, 3 classes': 37, 'stage 2 EfficientNet-B0': 9, 'direct SegFormer-B0, 5 classes': 25}


## Section 1: E4 on the validation photos

Question: do the ground truth, the cascade and the direct model give T_live, P_bleached and the KepMenLH category per photo from the same points, and how far are the two methods from the ground truth on the validation photos?

Checks: load the validation photos and the three models read only, put the same 30 grid points on every photo, label the points from the true masks, run the cascade and the direct model at the points, count the labels per photo and compute T_live, P_bleached and the category for the ground truth and both methods, print the MAE, the category agreement and the confusion matrices, and save the per-photo table under a new file name. Only the validation photos are used.

### Load the validation photos

In [4]:
raw_dir = base_dir + "/RawData"

import pandas as pd
from datasets import load_dataset

start = time.time()
validation = load_dataset("EPFL-ECEO/coralscapes", split="validation", cache_dir=raw_dir)
print(validation)
print("Loading took {:.1f} seconds".format(time.time() - start))
print("Validation photos: {}, expected 166: {}".format(len(validation), len(validation) == 166))

README.md:   0%|          | 0.00/13.7k [00:00<?, ?B/s]

Dataset({
    features: ['image', 'label'],
    num_rows: 166
})
Loading took 7.4 seconds
Validation photos: 166, expected 166: True


### Class names from the dataset card

Ids and names copied from notebook 07, which copied them from notebook 02.

In [5]:
class_names = {
    1: "seagrass",
    2: "trash",
    3: "other coral dead",
    4: "other coral bleached",
    5: "sand",
    6: "other coral alive",
    7: "human",
    8: "transect tools",
    9: "fish",
    10: "algae covered substrate",
    11: "other animal",
    12: "unknown hard substrate",
    13: "background",
    14: "dark",
    15: "transect line",
    16: "massive/meandering bleached",
    17: "massive/meandering alive",
    18: "rubble",
    19: "branching bleached",
    20: "branching dead",
    21: "millepora",
    22: "branching alive",
    23: "massive/meandering dead",
    24: "clam",
    25: "acropora alive",
    26: "sea cucumber",
    27: "turbinaria",
    28: "table acropora alive",
    29: "sponge",
    30: "anemone",
    31: "pocillopora alive",
    32: "table acropora dead",
    33: "meandering bleached",
    34: "stylophora alive",
    35: "sea urchin",
    36: "meandering alive",
    37: "meandering dead",
    38: "crown of thorn",
    39: "dead clam",
}

print("Classes on the card: {}".format(len(class_names)))

Classes on the card: 39


### Class groups

The coral ids are split into healthy, bleached and dead with the stage 2 state map of notebook 04. Substrate and the coral group come from the stage 1 map of notebook 02. The excluded coral ids come from `settings.json`. All of this is copied from notebook 09.

In [6]:
import json

# Stage 2 state map of notebook 04
healthy_ids = [6, 17, 22, 25, 28, 31, 34, 36]
bleached_ids = [4, 16, 19, 33]
dead_ids = [3, 20, 23, 32, 37]

# Stage 1 groups of notebook 02
substrate_ids = [5, 10, 12, 18]
coral_ids = [3, 4, 6, 16, 17, 19, 20, 21, 22, 23, 25, 27, 28, 31, 32, 33, 34, 36, 37]

with open(base_dir + "/Processed/stage2_patches/settings.json") as handle:
    settings = json.load(handle)
excluded_ids = settings["excluded_ids"]

print("Healthy ids: {}".format(healthy_ids))
print("Bleached ids: {}".format(bleached_ids))
print("Dead ids: {}".format(dead_ids))
print("Substrate ids: {}".format(substrate_ids))
print("Coral ids of notebook 02: {}".format(coral_ids))
print("Excluded ids from settings.json: {}".format(excluded_ids))

Healthy ids: [6, 17, 22, 25, 28, 31, 34, 36]
Bleached ids: [4, 16, 19, 33]
Dead ids: [3, 20, 23, 32, 37]
Substrate ids: [5, 10, 12, 18]
Coral ids of notebook 02: [3, 4, 6, 16, 17, 19, 20, 21, 22, 23, 25, 27, 28, 31, 32, 33, 34, 36, 37]
Excluded ids from settings.json: [21, 27]


### Build the 5-class mapping

In [7]:
group_names = {0: "coral healthy", 1: "coral bleached", 2: "coral dead", 3: "substrate", 4: "other"}
ignore_id = 255

# Everything on the card starts as other. Value 0 and the excluded coral ids are ignored
mapping = np.full(256, ignore_id, dtype=np.uint8)
for value in class_names:
    mapping[value] = 4
mapping[substrate_ids] = 3
mapping[healthy_ids] = 0
mapping[bleached_ids] = 1
mapping[dead_ids] = 2
mapping[excluded_ids] = ignore_id
mapping[0] = ignore_id

for group_id, group_name in group_names.items():
    names = [class_names[v] for v in class_names if mapping[v] == group_id]
    print("{} {} ({} ids): {}".format(group_id, group_name, len(names), names))
ignored = [class_names[v] for v in class_names if mapping[v] == ignore_id]
print("ignored ({} ids on the card): {}".format(len(ignored), ignored))
print("Mask value 0 maps to: {}".format(mapping[0]))

coral_five = set(healthy_ids + bleached_ids + dead_ids)
print("No id is in two classes: {}".format(
    len(healthy_ids + bleached_ids + dead_ids + substrate_ids) == len(coral_five | set(substrate_ids))))
print("Coral classes plus the ignored coral ids equal notebook 02's coral group: {}".format(
    coral_five | set(excluded_ids) == set(coral_ids)))
print("The excluded ids of settings.json are 21 and 27, as in the decision: {}".format(sorted(excluded_ids) == [21, 27]))
print("Class 4 holds the ids that are neither coral nor substrate in notebook 02: {}".format(
    {v for v in class_names if mapping[v] == 4} == set(class_names) - set(coral_ids) - set(substrate_ids)))

0 coral healthy (8 ids): ['other coral alive', 'massive/meandering alive', 'branching alive', 'acropora alive', 'table acropora alive', 'pocillopora alive', 'stylophora alive', 'meandering alive']
1 coral bleached (4 ids): ['other coral bleached', 'massive/meandering bleached', 'branching bleached', 'meandering bleached']
2 coral dead (5 ids): ['other coral dead', 'branching dead', 'massive/meandering dead', 'table acropora dead', 'meandering dead']
3 substrate (4 ids): ['sand', 'algae covered substrate', 'unknown hard substrate', 'rubble']
4 other (16 ids): ['seagrass', 'trash', 'human', 'transect tools', 'fish', 'other animal', 'background', 'dark', 'transect line', 'clam', 'sea cucumber', 'sponge', 'anemone', 'sea urchin', 'crown of thorn', 'dead clam']
ignored (2 ids on the card): ['millepora', 'turbinaria']
Mask value 0 maps to: 255
No id is in two classes: True
Coral classes plus the ignored coral ids equal notebook 02's coral group: True
The excluded ids of settings.json are 21 

### KepMenLH bands and the health numbers

Counts are in the order healthy, bleached, dead, substrate, other. Seabed points are coral and substrate points. T_live is healthy plus bleached over the seabed points, P_bleached is bleached over healthy plus bleached.

In [8]:
def kepmenlh_category(percent):
    for name, low, high in kepmenlh_bands[:-1]:
        if percent < high:
            return name
    return kepmenlh_bands[-1][0]


def health_numbers(counts):
    seabed = int(counts[0] + counts[1] + counts[2] + counts[3])
    live = int(counts[0] + counts[1])
    t_live = 100 * live / seabed if seabed > 0 else np.nan
    p_bleached = 100 * counts[1] / live if live > 0 else np.nan
    return seabed, t_live, p_bleached


label_names = ["healthy", "bleached", "dead", "substrate", "other"]

for percent in [0, 24.9, 25, 49.9, 50, 74.9, 75, 100]:
    print("{} percent live cover: {}".format(percent, kepmenlh_category(percent)))
example = np.array([10, 5, 5, 8, 2])
print("Example counts {}: seabed {}, T_live {:.1f}, P_bleached {:.1f}".format(example.tolist(), *health_numbers(example)))

0 percent live cover: Buruk
24.9 percent live cover: Buruk
25 percent live cover: Sedang
49.9 percent live cover: Sedang
50 percent live cover: Baik
74.9 percent live cover: Baik
75 percent live cover: Baik sekali
100 percent live cover: Baik sekali
Example counts [10, 5, 5, 8, 2]: seabed 28, T_live 53.6, P_bleached 33.3


### Stage 1 SegFormer-B0 with 3 classes, read only

In [9]:
import transformers
from transformers import SegformerForSemanticSegmentation

weights_name = "nvidia/mit-b0"
device = "cuda" if torch.cuda.is_available() else "cpu"
group_names3 = {0: "coral", 1: "substrate", 2: "other"}

stage1_state = torch.load(base_dir + "/Checkpoints/stage1_segformer_b0/best.pt", map_location="cpu", weights_only=False)
stage1_model = SegformerForSemanticSegmentation.from_pretrained(
    weights_name,
    num_labels=len(group_names3),
    id2label=group_names3,
    label2id={name: group_id for group_id, name in group_names3.items()},
    ignore_mismatched_sizes=True,
)
stage1_model.load_state_dict(stage1_state["model"])
stage1_model = stage1_model.to(device).eval()

expected = expected_epochs["stage 1 SegFormer-B0, 3 classes"]
print("transformers version: {}".format(transformers.__version__))
print("Stage 1 checkpoint holds epoch {}, expected {}: {}".format(stage1_state["epoch"], expected, stage1_state["epoch"] == expected))
print("Stage 1 parameters: {}".format(sum(p.numel() for p in stage1_model.parameters())))
del stage1_state

config.json:   0%|          | 0.00/70.0k [00:00<?, ?B/s]

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 14.4MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.bias                                         | UNEXPECTED | 
classifier.weight                                       | UNEXPECTED | 
decode_head.batch_norm.running_var                      | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.linear_fuse.weight                          | MISSING    | 
decode_head.batch_norm.weight                           

transformers version: 5.22.1
Stage 1 checkpoint holds epoch 37, expected 37: True
Stage 1 parameters: 3714915


### Stage 2 EfficientNet-B0, read only

In [10]:
from torch import nn
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

stage2_state_names = {0: "healthy", 1: "bleached", 2: "dead"}
stage2_path = base_dir + "/Checkpoints/stage2_efficientnet_b0/" + stage2_source + "/best.pt"
stage2_state = torch.load(stage2_path, map_location="cpu", weights_only=False)

# Built as in notebook 05; the weights come from the checkpoint
stage2_model = efficientnet_b0(weights=None)
stage2_model.classifier[1] = nn.Linear(stage2_model.classifier[1].in_features, len(stage2_state_names))
stage2_model.load_state_dict(stage2_state["model"])
stage2_model = stage2_model.to(device).eval()

transforms = EfficientNet_B0_Weights.IMAGENET1K_V1.transforms()
stage2_mean = torch.tensor(transforms.mean, device=device).view(1, 3, 1, 1)
stage2_std = torch.tensor(transforms.std, device=device).view(1, 3, 1, 1)

expected = expected_epochs["stage 2 EfficientNet-B0"]
print("Stage 2 checkpoint: {}".format(stage2_path))
print("Stage 2 checkpoint was trained on the '{}' source, expected '{}': {}".format(
    stage2_state["source"], stage2_source, stage2_state["source"] == stage2_source))
print("Stage 2 checkpoint holds epoch {}, expected {}: {}".format(stage2_state["epoch"], expected, stage2_state["epoch"] == expected))
print("Mean {}, std {}".format(transforms.mean, transforms.std))
print("Same mean and std as the segmentation models: {}".format(
    list(transforms.mean) == image_mean_values and list(transforms.std) == image_std_values))
print("Stage 2 parameters: {}".format(sum(p.numel() for p in stage2_model.parameters())))
del stage2_state

model.safetensors: reconstructing file:   0%|          |  0.00B / 14.3MB            

model.safetensors: downloading bytes:           |  0.00B            

Stage 2 checkpoint: /content/drive/MyDrive/ReefScan-ID/Checkpoints/stage2_efficientnet_b0/both/best.pt
Stage 2 checkpoint was trained on the 'both' source, expected 'both': True
Stage 2 checkpoint holds epoch 9, expected 9: True
Mean [0.485, 0.456, 0.406], std [0.229, 0.224, 0.225]
Same mean and std as the segmentation models: True
Stage 2 parameters: 4011391


### Direct SegFormer-B0 with 5 classes, read only

In [11]:
direct_state = torch.load(base_dir + "/Checkpoints/stage1_direct5/best.pt", map_location="cpu", weights_only=False)
direct_model = SegformerForSemanticSegmentation.from_pretrained(
    weights_name,
    num_labels=len(group_names),
    id2label=group_names,
    label2id={name: group_id for group_id, name in group_names.items()},
    ignore_mismatched_sizes=True,
)
direct_model.load_state_dict(direct_state["model"])
direct_model = direct_model.to(device).eval()

expected = expected_epochs["direct SegFormer-B0, 5 classes"]
print("Direct checkpoint holds epoch {}, expected {}: {}".format(direct_state["epoch"], expected, direct_state["epoch"] == expected))
print("Direct parameters: {}".format(sum(p.numel() for p in direct_model.parameters())))
del direct_state

[transformers] You passed `num_labels=5` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                                     | Status     | 
--------------------------------------------------------+------------+-
classifier.bias                                         | UNEXPECTED | 
classifier.weight                                       | UNEXPECTED | 
decode_head.batch_norm.running_var                      | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.linear_projections.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.batch_norm.num_batches_tracked              | MISSING    | 
decode_head.classifier.weight                           | MISSING    | 
decode_head.batch_norm.bias                             | MISSING    | 
decode_head.batch_norm.running_mean                     | MISSING    | 
decode_head.linear_fuse.weight                          | MISSING    | 
decode_head.batch_norm.weight                           

Direct checkpoint holds epoch 25, expected 25: True
Direct parameters: 3715429


### Grid points on a photo

The grid covers the photo without a border of the same width that notebook 04 kept from the photo edge, so that every 128 px patch lies inside the photo. Each cell holds one random point. The points depend only on the photo index and the seed. The evaluation therefore describes the interior of the photo, not its outer 128 px border.

In [12]:
edge_margin = settings["margin"]
half = patch_crop // 2


def photo_points(photo_index):
    rng = np.random.default_rng(point_seed + photo_index)
    inner_height = photo_height - 2 * edge_margin
    inner_width = photo_width - 2 * edge_margin
    offsets = rng.random((grid_rows, grid_cols, 2))
    rows = np.arange(grid_rows).reshape(-1, 1) + offsets[:, :, 0]
    cols = np.arange(grid_cols).reshape(1, -1) + offsets[:, :, 1]
    ys = (edge_margin + rows * inner_height / grid_rows).astype(np.int64).ravel()
    xs = (edge_margin + cols * inner_width / grid_cols).astype(np.int64).ravel()
    return ys, xs


ys, xs = photo_points(0)
print("Margin from the photo edge, as in notebook 04: {} px".format(edge_margin))
print("Grid cell size: {:.1f} by {:.1f} px".format(
    (photo_height - 2 * edge_margin) / grid_rows, (photo_width - 2 * edge_margin) / grid_cols))
print("Points on photo 0: {}, rows {} to {}, columns {} to {}".format(len(ys), ys.min(), ys.max(), xs.min(), xs.max()))
print("Every patch lies inside the photo: {}".format(
    ys.min() - half >= 0 and xs.min() - half >= 0 and ys.max() + half <= photo_height and xs.max() + half <= photo_width))
print("The same photo gives the same points: {}".format(
    all(np.array_equal(a, b) for a, b in zip(photo_points(0), photo_points(0)))))

Margin from the photo edge, as in notebook 04: 128 px
Grid cell size: 153.6 by 298.7 px
Points on photo 0: 30, rows 134 to 877, columns 138 to 1814
Every patch lies inside the photo: True
The same photo gives the same points: True


### Predictions at the points

Both segmentation models get the photo resized to 512 by 1024, and their prediction is upsampled to the full photo size. The cascade keeps substrate and other as predicted and sends the points predicted as coral to stage 2 as 128 px patches cut from the full photo, resized and normalised as in notebook 05.

In [13]:
import torch.nn.functional as F

image_mean = torch.tensor(image_mean_values, device=device).view(1, 3, 1, 1)
image_std = torch.tensor(image_std_values, device=device).view(1, 3, 1, 1)


def full_resolution_map(model, pixel_values):
    logits = model(pixel_values=pixel_values).logits
    upsampled = F.interpolate(logits, size=(photo_height, photo_width), mode="bilinear", align_corners=False)
    return upsampled.argmax(dim=1)[0]


def stage2_labels(patches):
    images = torch.from_numpy(patches).permute(0, 3, 1, 2).float().div(255).to(device)
    images = F.interpolate(images, size=(patch_size, patch_size), mode="bilinear", align_corners=False)
    return stage2_model((images - stage2_mean) / stage2_std).argmax(dim=1).cpu().numpy()


def predict_photo(image, image_array, ys, xs):
    if len(ys) == 0:
        return np.zeros(0, dtype=np.int64), np.zeros(0, dtype=np.int64)

    resized = image.resize((val_width, val_height), Image.BILINEAR)
    pixel_values = torch.from_numpy(np.array(resized)).permute(2, 0, 1).float().div(255).unsqueeze(0).to(device)
    pixel_values = (pixel_values - image_mean) / image_std

    ys_tensor = torch.from_numpy(ys).to(device)
    xs_tensor = torch.from_numpy(xs).to(device)
    with torch.no_grad():
        stage1_at = full_resolution_map(stage1_model, pixel_values)[ys_tensor, xs_tensor].cpu().numpy()
        direct_at = full_resolution_map(direct_model, pixel_values)[ys_tensor, xs_tensor].cpu().numpy()

        # Substrate is label 3, other is label 4, coral points get the stage 2 label 0, 1 or 2
        cascade_at = np.where(stage1_at == 1, 3, 4)
        coral = stage1_at == 0
        if coral.any():
            patches = np.stack([image_array[y - half:y + half, x - half:x + half] for y, x in zip(ys[coral], xs[coral])])
            cascade_at[coral] = stage2_labels(patches)
    return cascade_at, direct_at


print("Prediction functions defined")

Prediction functions defined


### Run the protocol on the validation photos

The true label at each point comes from the full-resolution mask with the 5-class mapping. Points whose true label is ignored (mask value 0, Millepora, Turbinaria) are dropped for the ground truth and for both methods.

In [14]:
sources = ["truth", "cascade", "direct"]
rows = []
all_labels = {source: [] for source in sources}
point_frames = []
dropped_points = 0

start = time.time()
for photo in range(len(validation)):
    item = validation[photo]
    image = item["image"]
    image_array = np.array(image)
    mask = np.array(item["label"])

    ys, xs = photo_points(photo)
    truth = mapping[mask[ys, xs]]
    keep = truth != ignore_id
    dropped_points += int((~keep).sum())
    ys, xs, truth = ys[keep], xs[keep], truth[keep]

    cascade, direct = predict_photo(image, image_array, ys, xs)
    labels = {"truth": truth, "cascade": cascade, "direct": direct}

    record = {"split": "validation", "photo": photo, "points_scored": len(truth)}
    for source in sources:
        counts = np.bincount(labels[source], minlength=5)
        seabed, t_live, p_bleached = health_numbers(counts)
        for name, count in zip(label_names, counts):
            record["{}_{}".format(source, name)] = int(count)
        record[source + "_seabed"] = seabed
        record[source + "_t_live_pct"] = t_live
        record[source + "_p_bleached_pct"] = p_bleached
        record[source + "_category"] = kepmenlh_category(t_live) if seabed > 0 else ""
        all_labels[source].append(labels[source])
    rows.append(record)
    if len(truth) > 0:
        names = np.array(label_names)
        point_frames.append(pd.DataFrame({
            "split": "validation", "photo": photo, "y": ys, "x": xs,
            "truth": names[truth], "cascade": names[cascade], "direct": names[direct],
        }))

    if (photo + 1) % 20 == 0:
        print("{} of {} photos done, {:.0f} seconds".format(photo + 1, len(validation), time.time() - start))

photo_table = pd.DataFrame(rows)
point_table = pd.concat(point_frames, ignore_index=True)
point_labels = {source: np.concatenate(all_labels[source]) for source in sources}

print("Photos: {}, seconds: {:.0f}".format(len(photo_table), time.time() - start))
print("Grid points: {}, dropped because the true label is ignored: {}, scored: {}".format(
    len(validation) * grid_rows * grid_cols, dropped_points, len(point_labels["truth"])))
print("Scored points per photo: smallest {}, median {:.0f}, largest {}".format(
    photo_table["points_scored"].min(), photo_table["points_scored"].median(), photo_table["points_scored"].max()))
print("Photos with no scored point: {}".format(int((photo_table["points_scored"] == 0).sum())))

20 of 166 photos done, 7 seconds
40 of 166 photos done, 11 seconds
60 of 166 photos done, 14 seconds
80 of 166 photos done, 18 seconds
100 of 166 photos done, 21 seconds
120 of 166 photos done, 25 seconds
140 of 166 photos done, 29 seconds
160 of 166 photos done, 32 seconds
Photos: 166, seconds: 33
Grid points: 4980, dropped because the true label is ignored: 796, scored: 4184
Scored points per photo: smallest 10, median 26, largest 30
Photos with no scored point: 0


### Photos without seabed or live coral points

In [15]:
print("Photos in the table: {}".format(len(photo_table)))
for source in sources:
    no_seabed = int((photo_table[source + "_seabed"] == 0).sum())
    no_live = int(((photo_table[source + "_healthy"] + photo_table[source + "_bleached"]) == 0).sum())
    print("{}: {} photos with no seabed point (no T_live), {} photos with no healthy or bleached point (no P_bleached)".format(
        source, no_seabed, no_live))

Photos in the table: 166
truth: 2 photos with no seabed point (no T_live), 35 photos with no healthy or bleached point (no P_bleached)
cascade: 1 photos with no seabed point (no T_live), 25 photos with no healthy or bleached point (no P_bleached)
direct: 1 photos with no seabed point (no T_live), 27 photos with no healthy or bleached point (no P_bleached)


### Results table: validation

MAE is in percentage points. A photo counts for a method only when both the ground truth and the method have the value: T_live needs seabed points in both, P_bleached needs healthy or bleached points in both. The category agreement is the share of those photos where the category equals the ground truth's.

In [16]:
def mae_points(method, truth):
    both = ~(np.isnan(method) | np.isnan(truth))
    value = float(np.abs(method[both] - truth[both]).mean()) if both.any() else np.nan
    return value, int(both.sum())


truth_t = photo_table["truth_t_live_pct"].to_numpy(dtype=float)
truth_p = photo_table["truth_p_bleached_pct"].to_numpy(dtype=float)

results = []
for source in ["cascade", "direct"]:
    method_t = photo_table[source + "_t_live_pct"].to_numpy(dtype=float)
    method_p = photo_table[source + "_p_bleached_pct"].to_numpy(dtype=float)
    t_mae, t_used = mae_points(method_t, truth_t)
    p_mae, p_used = mae_points(method_p, truth_p)

    both_t = ~(np.isnan(method_t) | np.isnan(truth_t))
    same = photo_table.loc[both_t, source + "_category"] == photo_table.loc[both_t, "truth_category"]
    results.append({
        "method": source,
        "MAE T_live (points)": t_mae,
        "photos (T_live)": t_used,
        "MAE P_bleached (points)": p_mae,
        "photos (P_bleached)": p_used,
        "category agreement": float(same.mean()) if both_t.any() else np.nan,
        "photos (category)": int(both_t.sum()),
    })

print("E4 on the {} validation photos, the same {} grid points for the ground truth and both methods".format(
    len(photo_table), grid_rows * grid_cols))
print("The points lie in the interior of the photo, the outer {} px border is not sampled".format(edge_margin))
print(pd.DataFrame(results).round(3).to_string(index=False))

E4 on the 166 validation photos, the same 30 grid points for the ground truth and both methods
The points lie in the interior of the photo, the outer 128 px border is not sampled
 method  MAE T_live (points)  photos (T_live)  MAE P_bleached (points)  photos (P_bleached)  category agreement  photos (category)
cascade               10.330              164                    9.599                  128               0.683                164
 direct               12.513              164                    6.723                  127               0.665                164


### Point-level confusion matrices

In [17]:
for source in ["cascade", "direct"]:
    matrix = np.zeros((5, 5), dtype=np.int64)
    np.add.at(matrix, (point_labels["truth"], point_labels[source]), 1)
    print("{}: predicted against true label over {} scored points (rows true, columns predicted)".format(
        source, int(matrix.sum())))
    print(pd.DataFrame(matrix, index=label_names, columns=label_names).to_string())

cascade: predicted against true label over 4184 scored points (rows true, columns predicted)
           healthy  bleached  dead  substrate  other
healthy        462        11    45         43      6
bleached        22        33     3          8      3
dead            84         5   159        151      3
substrate       57         5    39       1330     36
other          107         4    30        338   1200
direct: predicted against true label over 4184 scored points (rows true, columns predicted)
           healthy  bleached  dead  substrate  other
healthy        433        11    74         40      9
bleached         5        59     0          4      1
dead           105         9   154        130      4
substrate       62        16   138       1184     67
other          121         8    34        263   1253


### Save the per-photo table

The table is built on the Colab disk first. The cell refuses to save if the file name already exists, and the file is opened in a mode that cannot overwrite.

In [18]:
import shutil

local_dir = "/content/e4"
out_dir = base_dir + "/Processed/e4"
os.makedirs(local_dir, exist_ok=True)
os.makedirs(out_dir, exist_ok=True)

photo_table.to_csv(local_dir + "/" + output_name, index=False)
print("Table built on the Colab disk: {} rows, {} columns".format(len(photo_table), len(photo_table.columns)))
print("Columns: {}".format(list(photo_table.columns)))

before = sorted(os.listdir(out_dir))
print("Files in {} before saving: {}".format(out_dir, before))
if output_name in before:
    print("NOT SAVED: {} already exists, nothing was copied".format(output_name))
else:
    # Mode xb refuses to open a file that already exists
    with open(local_dir + "/" + output_name, "rb") as source_file, open(out_dir + "/" + output_name, "xb") as target_file:
        shutil.copyfileobj(source_file, target_file)
    print("{}: {:.2f} MB saved".format(output_name, os.path.getsize(out_dir + "/" + output_name) / 1024 ** 2))

after = sorted(os.listdir(out_dir))
print("Files in the folder after saving: {}".format(after))
print("Files added: {}, files removed: {}".format(
    [name for name in after if name not in before], [name for name in before if name not in after]))

Table built on the Colab disk: 166 rows, 30 columns
Columns: ['split', 'photo', 'points_scored', 'truth_healthy', 'truth_bleached', 'truth_dead', 'truth_substrate', 'truth_other', 'truth_seabed', 'truth_t_live_pct', 'truth_p_bleached_pct', 'truth_category', 'cascade_healthy', 'cascade_bleached', 'cascade_dead', 'cascade_substrate', 'cascade_other', 'cascade_seabed', 'cascade_t_live_pct', 'cascade_p_bleached_pct', 'cascade_category', 'direct_healthy', 'direct_bleached', 'direct_dead', 'direct_substrate', 'direct_other', 'direct_seabed', 'direct_t_live_pct', 'direct_p_bleached_pct', 'direct_category']
Files in /content/drive/MyDrive/ReefScan-ID/Processed/e4 before saving: ['validation_point_labels.csv', 'validation_point_labels_60.csv', 'validation_points.csv', 'validation_points_60.csv']
NOT SAVED: validation_points.csv already exists, nothing was copied
Files in the folder after saving: ['validation_point_labels.csv', 'validation_point_labels_60.csv', 'validation_points.csv', 'validati

### Save the point-level table

One row per scored point: photo, coordinates, true label, cascade label and direct label. It holds the same points as the confusion matrices, so the matrices can be rebuilt and single photos looked at later. Same rules as the per-photo table: built on the Colab disk first, never overwrites.

In [19]:
point_name = "validation_point_labels.csv"
point_table.to_csv(local_dir + "/" + point_name, index=False)

rebuilt = {source: np.zeros((5, 5), dtype=np.int64) for source in ["cascade", "direct"]}
codes = {name: number for number, name in enumerate(label_names)}
for source in rebuilt:
    np.add.at(rebuilt[source], (point_table["truth"].map(codes).to_numpy(), point_table[source].map(codes).to_numpy()), 1)
    original = np.zeros((5, 5), dtype=np.int64)
    np.add.at(original, (point_labels["truth"], point_labels[source]), 1)
    print("{}: matrix rebuilt from the point table equals the matrix above: {}".format(source, np.array_equal(rebuilt[source], original)))
print("Point table: {} rows, {} scored points, rows per photo equal points_scored: {}".format(
    len(point_table), len(point_labels["truth"]),
    point_table.groupby("photo").size().reindex(photo_table["photo"]).fillna(0).astype(int).tolist() == photo_table["points_scored"].tolist()))

before = sorted(os.listdir(out_dir))
print("Files in {} before saving: {}".format(out_dir, before))
if point_name in before:
    print("NOT SAVED: {} already exists, nothing was copied".format(point_name))
else:
    with open(local_dir + "/" + point_name, "rb") as source_file, open(out_dir + "/" + point_name, "xb") as target_file:
        shutil.copyfileobj(source_file, target_file)
    print("{}: {:.2f} MB saved".format(point_name, os.path.getsize(out_dir + "/" + point_name) / 1024 ** 2))

after = sorted(os.listdir(out_dir))
print("Files in the folder after saving: {}".format(after))
print("Files added: {}, files removed: {}".format(
    [name for name in after if name not in before], [name for name in before if name not in after]))

cascade: matrix rebuilt from the point table equals the matrix above: True
direct: matrix rebuilt from the point table equals the matrix above: True
Point table: 4184 rows, 4184 scored points, rows per photo equal points_scored: True
Files in /content/drive/MyDrive/ReefScan-ID/Processed/e4 before saving: ['validation_point_labels.csv', 'validation_point_labels_60.csv', 'validation_points.csv', 'validation_points_60.csv']
NOT SAVED: validation_point_labels.csv already exists, nothing was copied
Files in the folder after saving: ['validation_point_labels.csv', 'validation_point_labels_60.csv', 'validation_points.csv', 'validation_points_60.csv']
Files added: [], files removed: []


**Observation**: We evaluated the cascade and direct 5-class SegFormer on 166 validation photos using the same grid points. The cascade had lower T_live MAE (10.296 vs 12.540 percentage points) and higher KepMenLH category agreement (68.3% vs 66.5%). However, the direct model had lower P_bleached MAE (6.723 vs 9.599 percentage points). Out of 4,980 sampled points, 4,184 were scored after ignoring excluded labels. Overall, the cascade performed better for live coral cover, while the direct model estimated bleaching more accurately. These are validation results, not final test results.

## Section 2: 60 points on the validation photos

Question: with 60 grid points per photo instead of 30, how do the MAE of T_live, the MAE of P_bleached and the KepMenLH category agreement of the cascade and the direct model change on the validation photos? This is the point density ablation of the plan. Section 1 with 30 points stays the primary result.

Checks: the same rules as Section 1 with a 6 by 10 grid over the same interior area, every patch inside the photo, the models and functions of Section 1 reused (so Section 1 has to run first in the same session), the dropped and scored point counts, the photos without seabed or live coral points, the results table and the confusion matrices, a table with the 30 and 60 point results side by side, and two new files saved without overwriting. Only the validation photos are used.

### Settings for Section 2

In [ ]:
points_per_photo_60 = 60
grid_rows_60 = 6
grid_cols_60 = 10
output_name_60 = "validation_points_60.csv"
point_name_60 = "validation_point_labels_60.csv"

print("Points per photo: {} = {} rows x {} columns, one random point per cell, seed {}".format(
    points_per_photo_60, grid_rows_60, grid_cols_60, point_seed))
print("Grid size equals the points per photo: {}".format(grid_rows_60 * grid_cols_60 == points_per_photo_60))
print("Unchanged from Section 1: stage 2 source {}, patch {} px, border {} px, seed {}".format(
    stage2_source, patch_crop, edge_margin, point_seed))
print("Points with an ignored true label are dropped for the ground truth and for both methods")
needed = ["validation", "stage1_model", "stage2_model", "direct_model", "predict_photo", "photo_table", "point_labels",
          "results", "local_dir", "out_dir", "shutil"]
print("Section 1 objects found: {}, missing: {}".format(
    all(name in globals() for name in needed), [name for name in needed if name not in globals()]))
print("Files to save in {}: {}, {}".format(out_dir, output_name_60, point_name_60))

Points per photo: 60 = 6 rows x 10 columns, one random point per cell, seed 0
Grid size equals the points per photo: True
Unchanged from Section 1: stage 2 source both, patch 128 px, border 128 px, seed 0
Points with an ignored true label are dropped for the ground truth and for both methods
Section 1 objects found: True, missing: []
Files to save in /content/drive/MyDrive/ReefScan-ID/Processed/e4: validation_points_60.csv, validation_point_labels_60.csv


### Grid points with a 6 by 10 grid

The same rule as `photo_points` in Section 1, with the grid shape as an argument so that the Section 1 settings stay as they are. With the 5 by 6 shape it has to give the Section 1 points.

In [ ]:
def photo_points_grid(photo_index, rows_n, cols_n):
    rng = np.random.default_rng(point_seed + photo_index)
    inner_height = photo_height - 2 * edge_margin
    inner_width = photo_width - 2 * edge_margin
    offsets = rng.random((rows_n, cols_n, 2))
    rows = np.arange(rows_n).reshape(-1, 1) + offsets[:, :, 0]
    cols = np.arange(cols_n).reshape(1, -1) + offsets[:, :, 1]
    ys = (edge_margin + rows * inner_height / rows_n).astype(np.int64).ravel()
    xs = (edge_margin + cols * inner_width / cols_n).astype(np.int64).ravel()
    return ys, xs


all_points = [photo_points_grid(photo, grid_rows_60, grid_cols_60) for photo in range(len(validation))]
all_ys = np.concatenate([points[0] for points in all_points])
all_xs = np.concatenate([points[1] for points in all_points])

print("Margin from the photo edge: {} px".format(edge_margin))
print("Grid cell size: {:.1f} by {:.1f} px".format(
    (photo_height - 2 * edge_margin) / grid_rows_60, (photo_width - 2 * edge_margin) / grid_cols_60))
print("Points on photo 0: {}, rows {} to {}, columns {} to {}".format(
    len(all_points[0][0]), all_points[0][0].min(), all_points[0][0].max(), all_points[0][1].min(), all_points[0][1].max()))
print("Points on every photo: {}".format(sorted({len(points[0]) for points in all_points})))
print("Every patch lies inside the photo, on all {} photos: {}".format(
    len(validation),
    all_ys.min() - half >= 0 and all_xs.min() - half >= 0 and all_ys.max() + half <= photo_height and all_xs.max() + half <= photo_width))
print("The same photo gives the same points: {}".format(
    all(np.array_equal(a, b) for a, b in zip(photo_points_grid(0, grid_rows_60, grid_cols_60), all_points[0]))))
print("With the 5 by 6 shape the function gives the Section 1 points on all photos: {}".format(
    all(np.array_equal(a, b) for photo in range(len(validation))
        for a, b in zip(photo_points_grid(photo, grid_rows, grid_cols), photo_points(photo)))))

Margin from the photo edge: 128 px
Grid cell size: 128.0 by 179.2 px
Points on photo 0: 60, rows 133 to 886, columns 150 to 1888
Points on every photo: [60]
Every patch lies inside the photo, on all 166 photos: True
The same photo gives the same points: True
With the 5 by 6 shape the function gives the Section 1 points on all photos: True


### Run the protocol on the validation photos with 60 points

The loop is the one of Section 1 with the 60 point grid and its own variable names, so that the Section 1 tables stay available for the comparison.

In [ ]:
rows_60 = []
all_labels_60 = {source: [] for source in sources}
point_frames_60 = []
dropped_points_60 = 0

start = time.time()
for photo in range(len(validation)):
    item = validation[photo]
    image = item["image"]
    image_array = np.array(image)
    mask = np.array(item["label"])

    ys, xs = photo_points_grid(photo, grid_rows_60, grid_cols_60)
    truth = mapping[mask[ys, xs]]
    keep = truth != ignore_id
    dropped_points_60 += int((~keep).sum())
    ys, xs, truth = ys[keep], xs[keep], truth[keep]

    cascade, direct = predict_photo(image, image_array, ys, xs)
    labels = {"truth": truth, "cascade": cascade, "direct": direct}

    record = {"split": "validation", "photo": photo, "points_scored": len(truth)}
    for source in sources:
        counts = np.bincount(labels[source], minlength=5)
        seabed, t_live, p_bleached = health_numbers(counts)
        for name, count in zip(label_names, counts):
            record["{}_{}".format(source, name)] = int(count)
        record[source + "_seabed"] = seabed
        record[source + "_t_live_pct"] = t_live
        record[source + "_p_bleached_pct"] = p_bleached
        record[source + "_category"] = kepmenlh_category(t_live) if seabed > 0 else ""
        all_labels_60[source].append(labels[source])
    rows_60.append(record)
    if len(truth) > 0:
        names = np.array(label_names)
        point_frames_60.append(pd.DataFrame({
            "split": "validation", "photo": photo, "y": ys, "x": xs,
            "truth": names[truth], "cascade": names[cascade], "direct": names[direct],
        }))

    if (photo + 1) % 20 == 0:
        print("{} of {} photos done, {:.0f} seconds".format(photo + 1, len(validation), time.time() - start))

photo_table_60 = pd.DataFrame(rows_60)
point_table_60 = pd.concat(point_frames_60, ignore_index=True)
point_labels_60 = {source: np.concatenate(all_labels_60[source]) for source in sources}

print("Photos: {}, seconds: {:.0f}".format(len(photo_table_60), time.time() - start))
print("Grid points: {}, dropped because the true label is ignored: {}, scored: {}".format(
    len(validation) * grid_rows_60 * grid_cols_60, dropped_points_60, len(point_labels_60["truth"])))
print("Scored points per photo: smallest {}, median {:.0f}, largest {}".format(
    photo_table_60["points_scored"].min(), photo_table_60["points_scored"].median(), photo_table_60["points_scored"].max()))
print("Photos with no scored point: {}".format(int((photo_table_60["points_scored"] == 0).sum())))

20 of 166 photos done, 3 seconds
40 of 166 photos done, 6 seconds
60 of 166 photos done, 8 seconds
80 of 166 photos done, 11 seconds
100 of 166 photos done, 14 seconds
120 of 166 photos done, 17 seconds
140 of 166 photos done, 20 seconds
160 of 166 photos done, 23 seconds
Photos: 166, seconds: 24
Grid points: 9960, dropped because the true label is ignored: 1482, scored: 8478
Scored points per photo: smallest 23, median 52, largest 60
Photos with no scored point: 0


### Photos without seabed or live coral points, 60 points

In [ ]:
print("Photos in the table: {}".format(len(photo_table_60)))
for source in sources:
    no_seabed = int((photo_table_60[source + "_seabed"] == 0).sum())
    no_live = int(((photo_table_60[source + "_healthy"] + photo_table_60[source + "_bleached"]) == 0).sum())
    print("{}: {} photos with no seabed point (no T_live), {} photos with no healthy or bleached point (no P_bleached)".format(
        source, no_seabed, no_live))

Photos in the table: 166
truth: 2 photos with no seabed point (no T_live), 23 photos with no healthy or bleached point (no P_bleached)
cascade: 1 photos with no seabed point (no T_live), 16 photos with no healthy or bleached point (no P_bleached)
direct: 1 photos with no seabed point (no T_live), 17 photos with no healthy or bleached point (no P_bleached)


### Results table: validation, 60 points

Same columns and rules as Section 1: MAE in percentage points, and a photo counts for a method only when both the ground truth and the method have the value.

In [ ]:
truth_t_60 = photo_table_60["truth_t_live_pct"].to_numpy(dtype=float)
truth_p_60 = photo_table_60["truth_p_bleached_pct"].to_numpy(dtype=float)

results_60 = []
for source in ["cascade", "direct"]:
    method_t = photo_table_60[source + "_t_live_pct"].to_numpy(dtype=float)
    method_p = photo_table_60[source + "_p_bleached_pct"].to_numpy(dtype=float)
    t_mae, t_used = mae_points(method_t, truth_t_60)
    p_mae, p_used = mae_points(method_p, truth_p_60)

    both_t = ~(np.isnan(method_t) | np.isnan(truth_t_60))
    same = photo_table_60.loc[both_t, source + "_category"] == photo_table_60.loc[both_t, "truth_category"]
    results_60.append({
        "method": source,
        "MAE T_live (points)": t_mae,
        "photos (T_live)": t_used,
        "MAE P_bleached (points)": p_mae,
        "photos (P_bleached)": p_used,
        "category agreement": float(same.mean()) if both_t.any() else np.nan,
        "photos (category)": int(both_t.sum()),
    })

print("E4 on the {} validation photos, the same {} grid points for the ground truth and both methods".format(
    len(photo_table_60), grid_rows_60 * grid_cols_60))
print("The points lie in the interior of the photo, the outer {} px border is not sampled".format(edge_margin))
print(pd.DataFrame(results_60).round(3).to_string(index=False))

E4 on the 166 validation photos, the same 60 grid points for the ground truth and both methods
The points lie in the interior of the photo, the outer 128 px border is not sampled
 method  MAE T_live (points)  photos (T_live)  MAE P_bleached (points)  photos (P_bleached)  category agreement  photos (category)
cascade                7.513              164                    9.279                  142               0.707                164
 direct               10.880              164                    5.968                  141               0.689                164


### Point-level confusion matrices, 60 points

In [ ]:
for source in ["cascade", "direct"]:
    matrix = np.zeros((5, 5), dtype=np.int64)
    np.add.at(matrix, (point_labels_60["truth"], point_labels_60[source]), 1)
    print("{}: predicted against true label over {} scored points (rows true, columns predicted)".format(
        source, int(matrix.sum())))
    print(pd.DataFrame(matrix, index=label_names, columns=label_names).to_string())

cascade: predicted against true label over 8478 scored points (rows true, columns predicted)
           healthy  bleached  dead  substrate  other
healthy        951        31    83         95     18
bleached        40        66     5          9      2
dead           165        11   339        320     13
substrate      112        10   100       2672     65
other          193         7    70        658   2443
direct: predicted against true label over 8478 scored points (rows true, columns predicted)
           healthy  bleached  dead  substrate  other
healthy        925        25   123         86     19
bleached        11       104     2          5      0
dead           211        37   335        258      7
substrate      124        25   296       2366    148
other          232        15    74        508   2542


### Comparison: 30 points and 60 points

The 30 point rows are the Section 1 results from this session, the 60 point rows are the ones above. Both come from the same photos, models and rules; only the grid differs.

In [ ]:
comparison = []
for source in ["cascade", "direct"]:
    for points, table, scored in [(points_per_photo, results, len(point_labels["truth"])),
                                  (points_per_photo_60, results_60, len(point_labels_60["truth"]))]:
        row = [entry for entry in table if entry["method"] == source][0]
        record = {"method": source, "points per photo": points, "scored points": scored}
        record.update({key: value for key, value in row.items() if key != "method"})
        comparison.append(record)

print("Section 1 and Section 2 results on the {} validation photos".format(len(photo_table_60)))
print(pd.DataFrame(comparison).round(3).to_string(index=False))

Section 1 and Section 2 results on the 166 validation photos
 method  points per photo  scored points  MAE T_live (points)  photos (T_live)  MAE P_bleached (points)  photos (P_bleached)  category agreement  photos (category)
cascade                30           4184               10.296              164                    9.599                  128               0.683                164
cascade                60           8478                7.513              164                    9.279                  142               0.707                164
 direct                30           4184               12.540              164                    6.723                  127               0.665                164
 direct                60           8478               10.880              164                    5.968                  141               0.689                164


### Save the 60 point tables

Same save as Section 1: built on the Colab disk first, the folder listed before and after, and a file name that already exists is never overwritten.

In [ ]:
photo_table_60.to_csv(local_dir + "/" + output_name_60, index=False)
print("Per-photo table built on the Colab disk: {} rows, {} columns".format(len(photo_table_60), len(photo_table_60.columns)))
print("Same columns as the Section 1 table: {}".format(list(photo_table_60.columns) == list(photo_table.columns)))

before = sorted(os.listdir(out_dir))
print("Files in {} before saving: {}".format(out_dir, before))
if output_name_60 in before:
    print("NOT SAVED: {} already exists, nothing was copied".format(output_name_60))
else:
    # Mode xb refuses to open a file that already exists
    with open(local_dir + "/" + output_name_60, "rb") as source_file, open(out_dir + "/" + output_name_60, "xb") as target_file:
        shutil.copyfileobj(source_file, target_file)
    print("{}: {:.2f} MB saved".format(output_name_60, os.path.getsize(out_dir + "/" + output_name_60) / 1024 ** 2))

after = sorted(os.listdir(out_dir))
print("Files in the folder after saving: {}".format(after))
print("Files added: {}, files removed: {}".format(
    [name for name in after if name not in before], [name for name in before if name not in after]))

Per-photo table built on the Colab disk: 166 rows, 30 columns
Same columns as the Section 1 table: True
Files in /content/drive/MyDrive/ReefScan-ID/Processed/e4 before saving: ['validation_point_labels.csv', 'validation_points.csv']
validation_points_60.csv: 0.02 MB saved
Files in the folder after saving: ['validation_point_labels.csv', 'validation_points.csv', 'validation_points_60.csv']
Files added: ['validation_points_60.csv'], files removed: []


### Save the 60 point point-level table

In [ ]:
point_table_60.to_csv(local_dir + "/" + point_name_60, index=False)

rebuilt_60 = {source: np.zeros((5, 5), dtype=np.int64) for source in ["cascade", "direct"]}
codes_60 = {name: number for number, name in enumerate(label_names)}
for source in rebuilt_60:
    np.add.at(rebuilt_60[source], (point_table_60["truth"].map(codes_60).to_numpy(), point_table_60[source].map(codes_60).to_numpy()), 1)
    original = np.zeros((5, 5), dtype=np.int64)
    np.add.at(original, (point_labels_60["truth"], point_labels_60[source]), 1)
    print("{}: matrix rebuilt from the point table equals the matrix above: {}".format(source, np.array_equal(rebuilt_60[source], original)))
print("Point table: {} rows, {} scored points, rows per photo equal points_scored: {}".format(
    len(point_table_60), len(point_labels_60["truth"]),
    point_table_60.groupby("photo").size().reindex(photo_table_60["photo"]).fillna(0).astype(int).tolist() == photo_table_60["points_scored"].tolist()))

before = sorted(os.listdir(out_dir))
print("Files in {} before saving: {}".format(out_dir, before))
if point_name_60 in before:
    print("NOT SAVED: {} already exists, nothing was copied".format(point_name_60))
else:
    with open(local_dir + "/" + point_name_60, "rb") as source_file, open(out_dir + "/" + point_name_60, "xb") as target_file:
        shutil.copyfileobj(source_file, target_file)
    print("{}: {:.2f} MB saved".format(point_name_60, os.path.getsize(out_dir + "/" + point_name_60) / 1024 ** 2))

after = sorted(os.listdir(out_dir))
print("Files in the folder after saving: {}".format(after))
print("Files added: {}, files removed: {}".format(
    [name for name in after if name not in before], [name for name in before if name not in after]))

cascade: matrix rebuilt from the point table equals the matrix above: True
direct: matrix rebuilt from the point table equals the matrix above: True
Point table: 8478 rows, 8478 scored points, rows per photo equal points_scored: True
Files in /content/drive/MyDrive/ReefScan-ID/Processed/e4 before saving: ['validation_point_labels.csv', 'validation_points.csv', 'validation_points_60.csv']
validation_point_labels_60.csv: 0.38 MB saved
Files in the folder after saving: ['validation_point_labels.csv', 'validation_point_labels_60.csv', 'validation_points.csv', 'validation_points_60.csv']
Files added: ['validation_point_labels_60.csv'], files removed: []


**Observation**: We compared 30 and 60 grid points on the same 166 validation photos. With 60 points, 8,478 of 9,960 sampled points were scored after ignoring excluded labels. The cascade T_live MAE dropped from 10.296 to 7.513 percentage points, while the direct model dropped from 12.540 to 10.880. P_bleached MAE also decreased for both models (cascade: 9.599 to 9.279; direct: 6.723 to 5.968). KepMenLH category agreement increased from 68.3% to 70.7% for the cascade and from 66.5% to 68.9% for the direct model. Overall, 60 points gave better validation results for both approaches, but these results are not from the held-out test set.

## Section 3: E4 on the 392 Coralscapes test photos, run once

This section is the single use of the test set. The protocol is the frozen one of the handoff (2026-11-19): the three models of Section 1, the same grid rules, the same dropped points and the same metrics. 30 points per photo is the primary result and 60 points per photo is the predefined secondary analysis. Nothing is tuned and no setting is chosen from these numbers.

Question: how far are the cascade and the direct 5-class model from the ground truth on T_live, P_bleached and the KepMenLH category on the 392 test photos?

Checks: the settings and the loaded models are the frozen ones, none of the four output files exists yet, the split has 392 photos, the dropped and scored point counts, the photos without seabed or live coral points, the results tables for 30 and 60 points, the confusion matrices, and four files saved without overwriting. Section 3 needs Section 1 run first in the same session. Section 2 is not needed.

### Settings for Section 3

In [20]:
test_grids = {30: (5, 6), 60: (6, 10)}
test_roles = {30: "primary result", 60: "secondary analysis"}
test_output_names = {30: ("test_points.csv", "test_point_labels.csv"), 60: ("test_points_60.csv", "test_point_labels_60.csv")}
frozen_epochs = {"stage 1 SegFormer-B0, 3 classes": 37, "stage 2 EfficientNet-B0": 9, "direct SegFormer-B0, 5 classes": 25}

print("Frozen protocol of the handoff, 2026-11-19")
print("Stage 2: source {}, epoch {}. Stage 1: epoch {}. Direct 5-class: epoch {}".format(
    stage2_source, frozen_epochs["stage 2 EfficientNet-B0"], frozen_epochs["stage 1 SegFormer-B0, 3 classes"],
    frozen_epochs["direct SegFormer-B0, 5 classes"]))
for density, (rows_n, cols_n) in test_grids.items():
    print("{} points per photo ({}): {} rows x {} columns, one random point per cell, seed {}".format(
        density, test_roles[density], rows_n, cols_n, point_seed))
print("Border not sampled: {} px, patch {} px resized to {} px".format(edge_margin, patch_crop, patch_size))
print("Points with an ignored true label are dropped for the ground truth and for both methods")
print("KepMenLH bands on percent live cover: {}".format(kepmenlh_bands))
print("Output files in {}: {}".format(out_dir, test_output_names))

print("Section 1 settings are the frozen ones: {}".format(
    (stage2_source, points_per_photo, grid_rows, grid_cols, point_seed) == ("both", 30, 5, 6, 0) and expected_epochs == frozen_epochs))
print("Each grid has as many cells as points: {}".format(all(rows_n * cols_n == density for density, (rows_n, cols_n) in test_grids.items())))

needed = ["stage1_model", "stage2_model", "direct_model", "stage2_path", "predict_photo", "mae_points", "photo_points",
          "raw_dir", "local_dir", "out_dir", "shutil", "label_names", "mapping", "ignore_id", "sources"]
print("Section 1 objects found: {}, missing: {}".format(
    all(name in globals() for name in needed), [name for name in needed if name not in globals()]))


def check_model(label, path, model, expected_epoch):
    state = torch.load(path, map_location="cpu", weights_only=False)
    loaded = model.state_dict()
    same_keys = set(state["model"].keys()) == set(loaded.keys())
    same_weights = same_keys and all(torch.equal(loaded[key].cpu(), value) for key, value in state["model"].items())
    print("{}: checkpoint epoch {}, expected {}: {}; weights equal to the loaded model: {}; eval mode: {}; source in the checkpoint: {}".format(
        label, state["epoch"], expected_epoch, state["epoch"] == expected_epoch, same_weights, not model.training,
        state.get("source", "not stored")))


check_model("Stage 1", base_dir + "/Checkpoints/stage1_segformer_b0/best.pt", stage1_model, frozen_epochs["stage 1 SegFormer-B0, 3 classes"])
check_model("Stage 2", stage2_path, stage2_model, frozen_epochs["stage 2 EfficientNet-B0"])
check_model("Direct", base_dir + "/Checkpoints/stage1_direct5/best.pt", direct_model, frozen_epochs["direct SegFormer-B0, 5 classes"])

Frozen protocol of the handoff, 2026-11-19
Stage 2: source both, epoch 9. Stage 1: epoch 37. Direct 5-class: epoch 25
30 points per photo (primary result): 5 rows x 6 columns, one random point per cell, seed 0
60 points per photo (secondary analysis): 6 rows x 10 columns, one random point per cell, seed 0
Border not sampled: 128 px, patch 128 px resized to 224 px
Points with an ignored true label are dropped for the ground truth and for both methods
KepMenLH bands on percent live cover: [('Buruk', 0.0, 25.0), ('Sedang', 25.0, 50.0), ('Baik', 50.0, 75.0), ('Baik sekali', 75.0, 100.0)]
Output files in /content/drive/MyDrive/ReefScan-ID/Processed/e4: {30: ('test_points.csv', 'test_point_labels.csv'), 60: ('test_points_60.csv', 'test_point_labels_60.csv')}
Section 1 settings are the frozen ones: True
Each grid has as many cells as points: True
Section 1 objects found: True, missing: []
Stage 1: checkpoint epoch 37, expected 37: True; weights equal to the loaded model: True; eval mode: True

### Check that the test set was not used before

The run leaves four files in `Processed/e4`. If any of them exists, the test run was already done: the cell says so and stops, and no photo is loaded and no model is run. The cell that runs the protocol makes the same check again.

In [21]:
test_files = [name for pair in test_output_names.values() for name in pair]


def existing_test_files():
    return [name for name in test_files if os.path.exists(out_dir + "/" + name)]


print("Output folder exists: {}".format(os.path.isdir(out_dir)))
print("Files that mark a finished test run: {}".format(test_files))
print("Already in the folder: {}".format(existing_test_files()))
if existing_test_files():
    print("The test run was already done. Nothing was loaded or run, so the test set is not used twice.")
    raise RuntimeError("The test run was already done, stopping")
print("None of the four files exists, the test run can go on")

Output folder exists: True
Files that mark a finished test run: ['test_points.csv', 'test_point_labels.csv', 'test_points_60.csv', 'test_point_labels_60.csv']
Already in the folder: []
None of the four files exists, the test run can go on


### Load the test photos

In [22]:
test_photos = load_dataset("EPFL-ECEO/coralscapes", split="test", cache_dir=raw_dir)
print(test_photos)
print("Test photos: {}, expected 392: {}".format(len(test_photos), len(test_photos) == 392))

Dataset({
    features: ['image', 'label'],
    num_rows: 392
})
Test photos: 392, expected 392: True


### The protocol as one function

The grid function is the one of Section 2, copied so that Section 2 does not have to run. With the 5 by 6 shape it has to give the Section 1 points. `run_protocol` is the loop of Section 1 with the photos, the split name and the grid shape as arguments, and nothing else changed.

In [23]:
def photo_points_grid(photo_index, rows_n, cols_n):
    rng = np.random.default_rng(point_seed + photo_index)
    inner_height = photo_height - 2 * edge_margin
    inner_width = photo_width - 2 * edge_margin
    offsets = rng.random((rows_n, cols_n, 2))
    rows = np.arange(rows_n).reshape(-1, 1) + offsets[:, :, 0]
    cols = np.arange(cols_n).reshape(1, -1) + offsets[:, :, 1]
    ys = (edge_margin + rows * inner_height / rows_n).astype(np.int64).ravel()
    xs = (edge_margin + cols * inner_width / cols_n).astype(np.int64).ravel()
    return ys, xs


def run_protocol(photos, split_name, rows_n, cols_n):
    rows = []
    all_labels = {source: [] for source in sources}
    point_frames = []
    dropped_points = 0

    start = time.time()
    for photo in range(len(photos)):
        item = photos[photo]
        image = item["image"]
        image_array = np.array(image)
        mask = np.array(item["label"])

        ys, xs = photo_points_grid(photo, rows_n, cols_n)
        truth = mapping[mask[ys, xs]]
        keep = truth != ignore_id
        dropped_points += int((~keep).sum())
        ys, xs, truth = ys[keep], xs[keep], truth[keep]

        cascade, direct = predict_photo(image, image_array, ys, xs)
        labels = {"truth": truth, "cascade": cascade, "direct": direct}

        record = {"split": split_name, "photo": photo, "points_scored": len(truth)}
        for source in sources:
            counts = np.bincount(labels[source], minlength=5)
            seabed, t_live, p_bleached = health_numbers(counts)
            for name, count in zip(label_names, counts):
                record["{}_{}".format(source, name)] = int(count)
            record[source + "_seabed"] = seabed
            record[source + "_t_live_pct"] = t_live
            record[source + "_p_bleached_pct"] = p_bleached
            record[source + "_category"] = kepmenlh_category(t_live) if seabed > 0 else ""
            all_labels[source].append(labels[source])
        rows.append(record)
        if len(truth) > 0:
            names = np.array(label_names)
            point_frames.append(pd.DataFrame({
                "split": split_name, "photo": photo, "y": ys, "x": xs,
                "truth": names[truth], "cascade": names[cascade], "direct": names[direct],
            }))

        if (photo + 1) % 20 == 0:
            print("{} of {} photos done, {:.0f} seconds".format(photo + 1, len(photos), time.time() - start))

    photo_table = pd.DataFrame(rows)
    point_table = pd.concat(point_frames, ignore_index=True)
    point_labels = {source: np.concatenate(all_labels[source]) for source in sources}
    return photo_table, point_table, point_labels, dropped_points


print("The 5 by 6 shape gives the Section 1 points on all {} test photos: {}".format(
    len(test_photos),
    all(np.array_equal(a, b) for photo in range(len(test_photos))
        for a, b in zip(photo_points_grid(photo, grid_rows, grid_cols), photo_points(photo)))))
all_ys = np.concatenate([photo_points_grid(photo, rows_n, cols_n)[0] for rows_n, cols_n in test_grids.values() for photo in range(len(test_photos))])
all_xs = np.concatenate([photo_points_grid(photo, rows_n, cols_n)[1] for rows_n, cols_n in test_grids.values() for photo in range(len(test_photos))])
print("Every patch lies inside the photo for both grids on all photos: {}".format(
    all_ys.min() - half >= 0 and all_xs.min() - half >= 0 and all_ys.max() + half <= photo_height and all_xs.max() + half <= photo_width))

The 5 by 6 shape gives the Section 1 points on all 392 test photos: True
Every patch lies inside the photo for both grids on all photos: True


### Run the protocol with 30 points (primary result)

The only run of the cascade and the direct model on the test photos with the 30 point grid.

In [24]:
if existing_test_files():
    print("The test run was already done. Nothing was run, so the test set is not used twice.")
    raise RuntimeError("The test run was already done, stopping")

test_run = {}
density = 30
start = time.time()
table, points, labels, dropped = run_protocol(test_photos, "test", *test_grids[density])
test_run[density] = {"photo_table": table, "point_table": points, "point_labels": labels, "dropped": dropped}
print("Photos: {}, seconds: {:.0f}".format(len(table), time.time() - start))

20 of 392 photos done, 4 seconds
40 of 392 photos done, 9 seconds
60 of 392 photos done, 13 seconds
80 of 392 photos done, 16 seconds
100 of 392 photos done, 20 seconds
120 of 392 photos done, 24 seconds
140 of 392 photos done, 29 seconds
160 of 392 photos done, 34 seconds
180 of 392 photos done, 39 seconds
200 of 392 photos done, 42 seconds
220 of 392 photos done, 45 seconds
240 of 392 photos done, 50 seconds
260 of 392 photos done, 53 seconds
280 of 392 photos done, 59 seconds
300 of 392 photos done, 64 seconds
320 of 392 photos done, 67 seconds
340 of 392 photos done, 70 seconds
360 of 392 photos done, 73 seconds
380 of 392 photos done, 78 seconds
Photos: 392, seconds: 80


### Run the protocol with 60 points (secondary analysis)

In [25]:
if existing_test_files():
    print("The test run was already done. Nothing was run, so the test set is not used twice.")
    raise RuntimeError("The test run was already done, stopping")

density = 60
start = time.time()
table, points, labels, dropped = run_protocol(test_photos, "test", *test_grids[density])
test_run[density] = {"photo_table": table, "point_table": points, "point_labels": labels, "dropped": dropped}
print("Photos: {}, seconds: {:.0f}".format(len(table), time.time() - start))

20 of 392 photos done, 3 seconds
40 of 392 photos done, 7 seconds
60 of 392 photos done, 10 seconds
80 of 392 photos done, 14 seconds
100 of 392 photos done, 18 seconds
120 of 392 photos done, 21 seconds
140 of 392 photos done, 25 seconds
160 of 392 photos done, 29 seconds
180 of 392 photos done, 32 seconds
200 of 392 photos done, 36 seconds
220 of 392 photos done, 40 seconds
240 of 392 photos done, 44 seconds
260 of 392 photos done, 47 seconds
280 of 392 photos done, 50 seconds
300 of 392 photos done, 55 seconds
320 of 392 photos done, 58 seconds
340 of 392 photos done, 61 seconds
360 of 392 photos done, 64 seconds
380 of 392 photos done, 69 seconds
Photos: 392, seconds: 71


### Point counts and photos without seabed or live coral points

In [26]:
for density, run in test_run.items():
    table = run["photo_table"]
    rows_n, cols_n = test_grids[density]
    print("--- {} points per photo ({}) ---".format(density, test_roles[density]))
    print("Grid points: {}, dropped because the true label is ignored: {}, scored: {}".format(
        len(test_photos) * rows_n * cols_n, run["dropped"], len(run["point_labels"]["truth"])))
    print("Scored points per photo: smallest {}, median {:.0f}, largest {}".format(
        table["points_scored"].min(), table["points_scored"].median(), table["points_scored"].max()))
    print("Photos with no scored point: {}".format(int((table["points_scored"] == 0).sum())))
    print("Photos in the table: {}".format(len(table)))
    for source in sources:
        no_seabed = int((table[source + "_seabed"] == 0).sum())
        no_live = int(((table[source + "_healthy"] + table[source + "_bleached"]) == 0).sum())
        print("{}: {} photos with no seabed point (no T_live), {} photos with no healthy or bleached point (no P_bleached)".format(
            source, no_seabed, no_live))

--- 30 points per photo (primary result) ---
Grid points: 11760, dropped because the true label is ignored: 2475, scored: 9285
Scored points per photo: smallest 11, median 24, largest 30
Photos with no scored point: 0
Photos in the table: 392
truth: 10 photos with no seabed point (no T_live), 108 photos with no healthy or bleached point (no P_bleached)
cascade: 14 photos with no seabed point (no T_live), 101 photos with no healthy or bleached point (no P_bleached)
direct: 10 photos with no seabed point (no T_live), 115 photos with no healthy or bleached point (no P_bleached)
--- 60 points per photo (secondary analysis) ---
Grid points: 23520, dropped because the true label is ignored: 5049, scored: 18471
Scored points per photo: smallest 20, median 48, largest 60
Photos with no scored point: 0
Photos in the table: 392
truth: 9 photos with no seabed point (no T_live), 85 photos with no healthy or bleached point (no P_bleached)
cascade: 11 photos with no seabed point (no T_live), 79 phot

### Results tables: test, 30 points and 60 points

Same columns and rules as Sections 1 and 2: MAE in percentage points, and a photo counts for a method only when both the ground truth and the method have the value.

In [27]:
test_results = {}
for density, run in test_run.items():
    table = run["photo_table"]
    truth_t = table["truth_t_live_pct"].to_numpy(dtype=float)
    truth_p = table["truth_p_bleached_pct"].to_numpy(dtype=float)

    density_results = []
    for source in ["cascade", "direct"]:
        method_t = table[source + "_t_live_pct"].to_numpy(dtype=float)
        method_p = table[source + "_p_bleached_pct"].to_numpy(dtype=float)
        t_mae, t_used = mae_points(method_t, truth_t)
        p_mae, p_used = mae_points(method_p, truth_p)

        both_t = ~(np.isnan(method_t) | np.isnan(truth_t))
        same = table.loc[both_t, source + "_category"] == table.loc[both_t, "truth_category"]
        density_results.append({
            "method": source,
            "MAE T_live (points)": t_mae,
            "photos (T_live)": t_used,
            "MAE P_bleached (points)": p_mae,
            "photos (P_bleached)": p_used,
            "category agreement": float(same.mean()) if both_t.any() else np.nan,
            "photos (category)": int(both_t.sum()),
        })
    test_results[density] = density_results

    print("E4 on the {} test photos, {} points per photo ({}), the same points for the ground truth and both methods".format(
        len(table), density, test_roles[density]))
    print("The points lie in the interior of the photo, the outer {} px border is not sampled".format(edge_margin))
    print(pd.DataFrame(density_results).round(3).to_string(index=False))
    print()

E4 on the 392 test photos, 30 points per photo (primary result), the same points for the ground truth and both methods
The points lie in the interior of the photo, the outer 128 px border is not sampled
 method  MAE T_live (points)  photos (T_live)  MAE P_bleached (points)  photos (P_bleached)  category agreement  photos (category)
cascade                6.426              377                    5.078                  267               0.793                377
 direct                7.110              379                    3.600                  260               0.744                379

E4 on the 392 test photos, 60 points per photo (secondary analysis), the same points for the ground truth and both methods
The points lie in the interior of the photo, the outer 128 px border is not sampled
 method  MAE T_live (points)  photos (T_live)  MAE P_bleached (points)  photos (P_bleached)  category agreement  photos (category)
cascade                5.604              379                    

### Point-level confusion matrices, 30 points and 60 points

In [28]:
for density, run in test_run.items():
    for source in ["cascade", "direct"]:
        matrix = np.zeros((5, 5), dtype=np.int64)
        np.add.at(matrix, (run["point_labels"]["truth"], run["point_labels"][source]), 1)
        print("{} points, {}: predicted against true label over {} scored points (rows true, columns predicted)".format(
            density, source, int(matrix.sum())))
        print(pd.DataFrame(matrix, index=label_names, columns=label_names).to_string())
        print()

30 points, cascade: predicted against true label over 9285 scored points (rows true, columns predicted)
           healthy  bleached  dead  substrate  other
healthy       1069        10    73        121     65
bleached        38        93    15         16      1
dead            57         3   112        172      6
substrate      123         7   203       3799    196
other          117         0    25        357   2607

30 points, direct: predicted against true label over 9285 scored points (rows true, columns predicted)
           healthy  bleached  dead  substrate  other
healthy       1031        11    94        149     53
bleached         5       136    13          9      0
dead            42         6   147        153      2
substrate       92        26   398       3601    211
other          133        15    86        379   2493

60 points, cascade: predicted against true label over 18471 scored points (rows true, columns predicted)
           healthy  bleached  dead  substrate  oth

### Comparison: 30 points and 60 points on the test photos

Same layout as the comparison table of Section 2.

In [29]:
test_comparison = []
for source in ["cascade", "direct"]:
    for density, run in test_run.items():
        row = [entry for entry in test_results[density] if entry["method"] == source][0]
        record = {"method": source, "points per photo": density, "scored points": len(run["point_labels"]["truth"])}
        record.update({key: value for key, value in row.items() if key != "method"})
        test_comparison.append(record)

print("E4 results on the {} test photos".format(len(test_photos)))
print(pd.DataFrame(test_comparison).round(3).to_string(index=False))

E4 results on the 392 test photos
 method  points per photo  scored points  MAE T_live (points)  photos (T_live)  MAE P_bleached (points)  photos (P_bleached)  category agreement  photos (category)
cascade                30           9285                6.426              377                    5.078                  267               0.793                377
cascade                60          18471                5.604              379                    4.378                  297               0.826                379
 direct                30           9285                7.110              379                    3.600                  260               0.744                379
 direct                60          18471                6.594              382                    4.133                  292               0.788                382


### Save the per-photo tables

Same save as Sections 1 and 2: built on the Colab disk first, the folder listed before and after, and a file name that already exists is never overwritten.

In [30]:
os.makedirs(local_dir, exist_ok=True)
for density, run in test_run.items():
    name = test_output_names[density][0]
    run["photo_table"].to_csv(local_dir + "/" + name, index=False)
    print("{} points: per-photo table built on the Colab disk: {} rows, {} columns".format(
        density, len(run["photo_table"]), len(run["photo_table"].columns)))

    before = sorted(os.listdir(out_dir))
    if name in before:
        print("NOT SAVED: {} already exists, nothing was copied".format(name))
    else:
        # Mode xb refuses to open a file that already exists
        with open(local_dir + "/" + name, "rb") as source_file, open(out_dir + "/" + name, "xb") as target_file:
            shutil.copyfileobj(source_file, target_file)
        print("{}: {:.2f} MB saved".format(name, os.path.getsize(out_dir + "/" + name) / 1024 ** 2))
    after = sorted(os.listdir(out_dir))
    print("Files added: {}, files removed: {}".format(
        [file for file in after if file not in before], [file for file in before if file not in after]))

30 points: per-photo table built on the Colab disk: 392 rows, 30 columns
test_points.csv: 0.04 MB saved
Files added: ['test_points.csv'], files removed: []
60 points: per-photo table built on the Colab disk: 392 rows, 30 columns
test_points_60.csv: 0.05 MB saved
Files added: ['test_points_60.csv'], files removed: []


### Save the point-level tables and rebuild the confusion matrices

The matrices are rebuilt from the point tables as a check before the tables are saved.

In [31]:
codes_test = {name: number for number, name in enumerate(label_names)}
for density, run in test_run.items():
    name = test_output_names[density][1]
    points = run["point_table"]
    table = run["photo_table"]
    points.to_csv(local_dir + "/" + name, index=False)

    for source in ["cascade", "direct"]:
        rebuilt = np.zeros((5, 5), dtype=np.int64)
        np.add.at(rebuilt, (points["truth"].map(codes_test).to_numpy(), points[source].map(codes_test).to_numpy()), 1)
        original = np.zeros((5, 5), dtype=np.int64)
        np.add.at(original, (run["point_labels"]["truth"], run["point_labels"][source]), 1)
        print("{} points, {}: matrix rebuilt from the point table equals the matrix above: {}".format(
            density, source, np.array_equal(rebuilt, original)))
    print("{} points: point table {} rows, {} scored points, rows per photo equal points_scored: {}".format(
        density, len(points), len(run["point_labels"]["truth"]),
        points.groupby("photo").size().reindex(table["photo"]).fillna(0).astype(int).tolist() == table["points_scored"].tolist()))

    before = sorted(os.listdir(out_dir))
    if name in before:
        print("NOT SAVED: {} already exists, nothing was copied".format(name))
    else:
        with open(local_dir + "/" + name, "rb") as source_file, open(out_dir + "/" + name, "xb") as target_file:
            shutil.copyfileobj(source_file, target_file)
        print("{}: {:.2f} MB saved".format(name, os.path.getsize(out_dir + "/" + name) / 1024 ** 2))
    after = sorted(os.listdir(out_dir))
    print("Files added: {}, files removed: {}".format(
        [file for file in after if file not in before], [file for file in before if file not in after]))

print("Files in {}: {}".format(out_dir, sorted(os.listdir(out_dir))))

30 points, cascade: matrix rebuilt from the point table equals the matrix above: True
30 points, direct: matrix rebuilt from the point table equals the matrix above: True
30 points: point table 9285 rows, 9285 scored points, rows per photo equal points_scored: True
test_point_labels.csv: 0.37 MB saved
Files added: ['test_point_labels.csv'], files removed: []
60 points, cascade: matrix rebuilt from the point table equals the matrix above: True
60 points, direct: matrix rebuilt from the point table equals the matrix above: True
60 points: point table 18471 rows, 18471 scored points, rows per photo equal points_scored: True
test_point_labels_60.csv: 0.74 MB saved
Files added: ['test_point_labels_60.csv'], files removed: []
Files in /content/drive/MyDrive/ReefScan-ID/Processed/e4: ['test_point_labels.csv', 'test_point_labels_60.csv', 'test_points.csv', 'test_points_60.csv', 'validation_point_labels.csv', 'validation_point_labels_60.csv', 'validation_points.csv', 'validation_points_60.csv']

**Observation**: We evaluated both models on 392 held-out Coralscapes test photos. With 30 points per photo, 9,285 points were scored. The cascade had lower T_live MAE (6.426 vs 7.110 percentage points) and higher KepMenLH category agreement (79.3% vs 74.4%), while the direct model had lower P_bleached MAE (3.600 vs 5.078 percentage points). With 60 points, T_live MAE and category agreement improved for both models, but the direct model's P_bleached MAE increased from 3.600 to 4.133. Overall, the cascade performed better for live coral cover and reef health categories, while the direct model estimated bleaching more accurately. These are the final held-out test results for the sampled interior points.